### Create binary labels for artificial disease, save dataloader table, and adapt base split

In [ ]:
import pandas as pd
import json

In [ ]:
# cidp 1 for artificial disease
field_1 = 27359
max_pc_1 = 426
table_path_1 = f"/sc-projects/sc-proj-cc15-cn-ukbiobank/analyses/Explanation-benchmark-paper/model_training/training_files/aparc2009/table_{field_1}_{max_pc_1}.csv"

# cidp 2 for artificial disease
field_2 = 26559
max_pc_2 = 26
table_path_2 = f"/sc-projects/sc-proj-cc15-cn-ukbiobank/analyses/Explanation-benchmark-paper/model_training/training_files/aseg_prototype/table_{field_2}_{max_pc_2}.csv"

# ukbb table path
ukb_table_path = "/sc-resources/ukb/data/projects/33073/ukb_data/table/ukb.parquet"

In [ ]:
# read cidp tables with erids and targets
table_df_1 = pd.read_csv(table_path_1, index_col=0)
table_df_2 = pd.read_csv(table_path_2, index_col=0)
merged_df = pd.merge(table_df_1, table_df_2, left_on="eid", right_on="eid", how='inner')


# create binary labels for artificial disease
merged_df["marker_1"] = None
merged_df["marker_2"] = None

lower_bound_idx = int((0.4)*len(merged_df))

target_1_sorted_df = merged_df.sort_values(by=["target_x"], ascending=False).reset_index()
target_1_sorted_df.loc[:lower_bound_idx, "marker_1"] = 1
target_1_sorted_df.loc[lower_bound_idx:int(1.5*lower_bound_idx), "marker_1"] = -1
target_1_sorted_df.loc[int(1.5*lower_bound_idx):, "marker_1"] = 0


target_2_sorted_df = target_1_sorted_df.sort_values(by=["target_y"], ascending=False).reset_index()
target_2_sorted_df.loc[:lower_bound_idx, "marker_2"] = 1
target_2_sorted_df.loc[lower_bound_idx:int(1.5*lower_bound_idx), "marker_2"] = - 1
target_2_sorted_df.loc[int(1.5*lower_bound_idx):, "marker_2"] = 0


target_2_sorted_df = target_2_sorted_df.drop(target_2_sorted_df[target_2_sorted_df.marker_1 == -1].index)
target_2_sorted_df = target_2_sorted_df.drop(target_2_sorted_df[target_2_sorted_df.marker_2 == -1].index)

target_2_sorted_df["target"] = 0
target_2_sorted_df.loc[(target_2_sorted_df['marker_1'] == 1) & (target_2_sorted_df['marker_2'] == 0), "target"] = 1

target_2_sorted_df = target_2_sorted_df[['eid', 'raw_x', 'coefs_to_mni_x', 'target']]
target_2_sorted_df = target_2_sorted_df.rename(columns={'raw_x': 'raw', 'coefs_to_mni_x': 'coefs_to_mni'})

# save dataloader table
target_2_sorted_df.to_csv(f"/sc-projects/sc-proj-cc15-cn-ukbiobank/analyses/Explanation-benchmark-paper/model_training/training_files/dummy_diseases/high_{field_1}_{max_pc_1}_low_{field_2}_{max_pc_2}_0.64ds.csv")

print(len(target_2_sorted_df))
print(len(target_2_sorted_df.loc[target_2_sorted_df["target"] == 1]))
print(len(target_2_sorted_df.loc[target_2_sorted_df["target"] == 0]))

adapt base split:

In [ ]:
old_split_path = '/sc-projects/sc-proj-cc15-cn-ukbiobank/analyses/Explanation-benchmark-paper/model_training/training_files/simple_split_aseg.json'
new_split_path = f'/sc-projects/sc-proj-cc15-cn-ukbiobank/analyses/Explanation-benchmark-paper/model_training/training_files/dummy_diseases/high_{field_1}_{max_pc_1}_low_{field_2}_{max_pc_2}_0.64ds.json'

with open(old_split_path, 'r') as file:
    # Load JSON data into a Python dictionary
    original_split = json.load(file)


eids = target_2_sorted_df['eid'].values

# eids to int
int_eids = []
for eid in eids:
  int_eids.append(int(eid))


for set in ['train', 'val', 'test', 'augmentation']:
    print(len(original_split[set]))    
    shared_eids = [item for item in original_split[set] if item in int_eids]
    print(len(shared_eids))

    original_split[set] = shared_eids

with open(new_split_path, "w") as f:
    json.dump(original_split, f)
